# 01 · Diagnose first, then handle a rare class

*Intermediate Model Training*, notebook 1 of 4 · lessons 1 and 2: **Diagnose the bottleneck before tuning** and **Handle imbalance and select a decision threshold**

A disappointing model can fail for very different reasons. It may be too simple for the pattern, it may have seen too few examples, or it may be judged with the wrong decision rule. In this notebook you first **locate the bottleneck** with a baseline and learning curves. Then you work on a problem in which only about one example in nine is positive, and you type the loss, the curve and the decision rule that such problems need. Along the way you will see that weighting the loss and choosing the threshold are two different tools.

## What you will build

- `prevalence_log_loss(y_train, y_eval)`: the loss of the no-skill baseline that every model must beat;
- `weighted_cross_entropy(logits, targets, class_weights)`: a class-weighted loss on the tiny tensor engine, gradient included;
- `precision_recall_curve(scores, y)`: precision and recall at every possible threshold, from scratch;
- `choose_threshold(scores, y)`: the threshold with the best F1 score on validation data, frozen before the test set is touched.

## Learning objectives

By the end of this notebook you will be able to:

1. compare a model with a no-skill baseline, and read learning curves over dataset size and over updates to tell a capacity problem from a data problem;
2. explain why accuracy misleads on a rare class, and implement class-weighted cross-entropy;
3. compute a precision–recall curve and its average precision from scores and labels;
4. choose a decision threshold on validation data, freeze it and evaluate it once on the test set;
5. separate a model's ranking, its probabilities and its decisions.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the data are generated in the notebook from fixed seeds, so your numbers should match the ones quoted in the text up to small differences between computers.
- The networks are trained with the tiny tensor engine and the small `MLP`, `Adam` and `fit_standardiser` from Base Model Training. Both are pasted into hidden recap cells below.
- The whole notebook runs in a few seconds on a laptop; in the browser it takes a little longer.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines the pieces from Base Model Training that this notebook trains with, under their usual names:

- `fit_standardiser(X)` returns the column means and scales of the **training** rows;
- `Linear` and `MLP(sizes, rng, dropout=0.0)` build a ReLU network; `model(Tensor(X))` returns logits of shape `(batch, classes)`, `model.parameters()` lists its weights, and `model.train()` / `model.eval()` switch dropout on and off;
- `Adam(params, lr)` updates the parameters with `opt.zero_grad()`, `loss.backward()` and `opt.step()`.

In [ ]:
# 📦 Recap: the pieces from Base Model Training, under their usual names.

def fit_standardiser(X):
    """Column means and scales of the TRAINING rows; apply them to every split as (X - mean) / scale."""
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


class Linear:
    """A dense layer x @ W + b, with He-initialised W of shape (n_in, n_out) and a zero bias."""

    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]


def dropout(x, p, training, rng):
    """Inverted dropout: while training, zero each unit with probability p and scale the rest by 1 / (1 - p)."""
    if not training or p == 0.0:
        return x
    keep = (rng.random(x.shape) >= p) / (1.0 - p)
    return x * keep


class MLP:
    """ReLU hidden layers and a linear output layer, e.g. MLP([6, 32, 2], rng). Call it on a Tensor (batch, n_in)."""

    def __init__(self, sizes, rng, dropout=0.0):
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.p = dropout
        self.training = True

    def __call__(self, x, rng=None):
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if index < len(self.layers) - 1:
                x = dropout(x.relu(), self.p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


class Adam:
    """Adam over a list of engine parameters: opt.zero_grad(), loss.backward(), opt.step()."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[...] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

## 1 · A generated problem with a rare positive class

Every row has six features drawn from a standard normal distribution. A hidden score combines them,

$$s = 1.5\,x_0 - x_1 + x_2 x_3 + 0.8\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1),$$

and a row is **positive** when $s > 2.8$, which happens for roughly one row in nine. Two details matter later. The product $x_2 x_3$ is an *interaction*: no weighted sum of the six features can express it, so a linear model will miss part of the pattern. The noise term means that even a perfect model cannot separate the classes completely.

We draw three independent sets: 1,000 training rows, 500 validation rows and 500 test rows. The standardiser is fitted on the training rows only and then applied to all three, so no statistic leaks from validation or test data into training.

In [ ]:
def make_rare_class_data(n, seed):
    """n rows of six standard-normal features; label 1 when a noisy score exceeds 2.8 (about 11% of rows)."""
    g = np.random.default_rng(seed)
    X = g.normal(size=(n, 6))
    score = 1.5 * X[:, 0] - X[:, 1] + X[:, 2] * X[:, 3] + 0.8 * g.normal(size=n)
    return X, (score > 2.8).astype(int)


X_train_raw, y_train = make_rare_class_data(1000, seed=11)
X_val_raw, y_val = make_rare_class_data(500, seed=22)
X_test_raw, y_test = make_rare_class_data(500, seed=33)

mean, scale = fit_standardiser(X_train_raw)  # training rows only
X_train, X_val, X_test = [(X - mean) / scale for X in (X_train_raw, X_val_raw, X_test_raw)]

for name, y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:>10}: {len(y):4d} rows, {y.sum():3d} positive ({y.mean():.1%})")
print(f"Always answering 'negative' would score {1 - y_val.mean():.1%} validation accuracy and find no positive at all.")

**What to notice.** About 11% of the rows are positive in each split. A "model" that always says *negative* is right about 89% of the time and completely useless, so accuracy is the wrong yardstick here. We will use log loss to judge the probabilities and precision and recall to judge the decisions.

## 2 · The baseline every model must beat

Before tuning anything, measure what *no skill* scores. The no-skill model knows only the class balance: it predicts the training prevalence

$$\pi = \frac{1}{m} \sum_{j=1}^{m} y^{\text{train}}_j$$

as the probability of *positive* for every example. Its log loss (binary cross-entropy) on evaluation labels $y_1, \dots, y_n$ is

$$L_{\text{base}} = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log \pi + (1 - y_i) \log (1 - \pi) \Big].$$

A trained classifier whose validation loss is not clearly below $L_{\text{base}}$ has learned nothing that generalises, however good its training loss looks.

**Shapes.** `y_train` has shape `(m,)` and `y_eval` has shape `(n,)`; both hold the integers 0 and 1. The result is one Python `float`.

In [ ]:
pi = y_train.mean()
print(f"training prevalence: pi = {pi:.3f}")
print(f"loss for one negative example: -log(1 - pi) = {-np.log(1 - pi):.3f}")
print(f"loss for one positive example: -log(pi)     = {-np.log(pi):.3f}")

### ✍️ Your turn: `prevalence_log_loss`

Write `prevalence_log_loss(y_train, y_eval)`:

- **inputs:** `y_train`, the 0/1 labels the prevalence is estimated from, and `y_eval`, the 0/1 labels the loss is measured on;
- **output:** the mean log loss of predicting $\pi$ for every example of `y_eval`, as a Python `float`.

Hints: clip the prevalence with `np.clip(np.mean(y_train), 1e-12, 1 - 1e-12)` so that a training set without positives does not produce $\log 0$; then follow the formula with `np.log` and `np.mean`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def prevalence_log_loss_reference(y_train, y_eval):
    pi = np.clip(np.mean(y_train), 1e-12, 1 - 1e-12)
    return float(-np.mean(y_eval * np.log(pi) + (1 - y_eval) * np.log(1 - pi)))

In [ ]:
def prevalence_log_loss(y_train, y_eval):
    """Log loss of always predicting the training prevalence. y_train: (m,) 0/1, y_eval: (n,) 0/1. Returns a float."""
    # ✍️ Your code here (two lines: the clipped prevalence, then the formula)
    raise NotImplementedError

In [ ]:
prevalence_log_loss = practice("prevalence_log_loss", prevalence_log_loss, prevalence_log_loss_reference, cases=[
    (np.array([0, 0, 0, 1]), np.array([1, 0])),
    (y_train, y_val),
    (np.array([0, 0, 0]), np.array([0, 1])),  # edge case: no positives in training, so the clip matters
])
baseline = prevalence_log_loss(y_train, y_val)
print(f"no-skill validation log loss: {baseline:.3f}")

**Expected behaviour.** With $\pi = 0.25$ the first case costs $-\tfrac{1}{2}(\log 0.25 + \log 0.75) \approx 0.837$. On our data the no-skill validation loss is about **0.32**: that is the number to beat. Without the clip, the third case would return `inf`, because the baseline would call a positive example impossible.

## 3 · Learning curves locate the bottleneck

A **learning curve** puts the training loss and the validation loss side by side while one thing changes: the number of training rows, or the number of updates. The pattern tells you which kind of fix can help.

| What you see | Likely bottleneck | Fixes that can help |
|---|---|---|
| Both losses high, close together, flat as data grows | capacity or features (high bias) | a bigger model, better features |
| Training loss far below validation loss, gap shrinking with more data | too little data for the model (high variance) | more data, regularisation, early stopping |
| Training loss itself not falling | optimisation | learning rate, initialisation, a bug in the loop |

Noisy labels or a validation set from a different source can imitate these patterns, so inspect the data as well as the curves.

We compare two models: a **linear** model, `MLP([6, 2])`, which is logistic regression, and an **MLP** with one hidden layer of 32 ReLU units. `fit` trains a model with Adam on random mini-batches and `mean_loss` measures cross-entropy in evaluation mode. Both are given, because they only call pieces you already know.

In [ ]:
def mean_loss(model, X, y):
    """Mean cross-entropy of a model on (X, y), in evaluation mode."""
    model.eval()
    return float(cross_entropy(model(Tensor(X)), y).data)


def fit(model, X, y, steps=300, lr=0.01, batch_size=128, seed=0, loss_fn=cross_entropy):
    """Train with Adam on random mini-batches drawn with a seeded generator; returns the model in eval mode."""
    opt = Adam(model.parameters(), lr=lr)
    g = np.random.default_rng(seed)
    for _ in range(steps):
        rows = g.integers(0, len(y), size=min(batch_size, len(y)))
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Tensor(X[rows]), g), y[rows])
        loss.backward()
        opt.step()
    model.eval()
    return model


sizes = [50, 100, 200, 400, 1000]
curves = {}
for name, hidden_layers in [("linear", []), ("MLP", [32])]:
    rows = []
    for n in sizes:
        model = fit(MLP([6, *hidden_layers, 2], np.random.default_rng(0)), X_train[:n], y_train[:n])
        rows.append((n, mean_loss(model, X_train[:n], y_train[:n]), mean_loss(model, X_val, y_val)))
    curves[name] = np.array(rows)

print(f"no-skill baseline on validation: {baseline:.3f}\n")
print(f"{'rows':>5} | {'linear: train':>13} {'val':>6} | {'MLP: train':>10} {'val':>6}")
for (n, lin_train, lin_val), (_, mlp_train, mlp_val) in zip(curves["linear"], curves["MLP"]):
    print(f"{int(n):5d} | {lin_train:13.3f} {lin_val:6.3f} | {mlp_train:10.3f} {mlp_val:6.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
for ax, name in zip(axes, ["linear", "MLP"]):
    ax.plot(curves[name][:, 0], curves[name][:, 1], "o-", label="training")
    ax.plot(curves[name][:, 0], curves[name][:, 2], "s-", label="validation")
    ax.axhline(baseline, color="grey", linestyle="--", label="no-skill baseline")
    ax.set_xscale("log"); ax.set_xlabel("training rows"); ax.set_title(f"{name} model")
axes[0].set_ylabel("cross-entropy"); axes[0].legend()
axes[1].set_ylim(0, 0.6)
plt.tight_layout(); plt.show()

**Reading the curves.**

- The **linear** model's training and validation losses sit close together, about 0.2, and barely move as the training set grows twentyfold. More data will not help it: it cannot represent the interaction $x_2 x_3$. That is a capacity bottleneck.
- The **MLP** trained on 50 rows almost memorises them (training loss near 0) while its validation loss is *worse than the no-skill baseline*. As rows are added the gap closes, and with 1,000 rows it reaches about 0.13, well below the linear model. Its bottleneck was data, not capacity.

The same validation number, say 0.2, can therefore call for opposite fixes. Without the training curve beside it you cannot tell which.

### Loss over updates, and where to stop

The second kind of curve follows one run over time. Below, a wide network (two hidden layers of 64 units) trains on only 150 rows with full-batch Adam. Watch the two losses part company: the training loss keeps falling towards zero, while the validation loss reaches a minimum and then climbs. **Early stopping** keeps the weights from the update with the lowest validation loss; in practice you copy them when the minimum is reached.

In [ ]:
small_X, small_y = X_train[:150], y_train[:150]
wide = MLP([6, 64, 64, 2], np.random.default_rng(0))
opt = Adam(wide.parameters(), lr=0.01)
history = []
for update in range(1, 301):
    wide.train()
    opt.zero_grad()
    loss = cross_entropy(wide(Tensor(small_X)), small_y)  # full batch: all 150 rows
    loss.backward()
    opt.step()
    if update % 10 == 0:
        history.append((update, mean_loss(wide, small_X, small_y), mean_loss(wide, X_val, y_val)))
history = np.array(history)
best_row = history[np.argmin(history[:, 2])]
print(f"lowest validation loss {best_row[2]:.3f} at update {int(best_row[0])}")
print(f"after 300 updates: training loss {history[-1, 1]:.4f}, validation loss {history[-1, 2]:.3f}")

plt.figure(figsize=(6, 3.4))
plt.plot(history[:, 0], history[:, 1], label="training")
plt.plot(history[:, 0], history[:, 2], label="validation")
plt.axvline(best_row[0], color="grey", linestyle=":", label="early-stopping point")
plt.xlabel("update"); plt.ylabel("cross-entropy"); plt.ylim(0, 1); plt.legend(); plt.show()

**What to notice.** The validation loss is lowest after a few dozen updates and then rises steadily, while the training loss heads towards zero. Training for longer makes this model *worse*. A controlled ablation would change one thing at a time from here (fewer hidden units, more rows, or weight decay) and keep the split, the seed and the metric fixed, so that any change in the curve can be traced to its cause.

## 4 · Class-weighted cross-entropy

With about 11% positives, the ordinary loss is dominated by negatives. The model's probabilities can still be honest estimates, but at the default threshold of 0.5 it will find only part of the positives. **Class weighting** multiplies each example's loss by the weight of its class:

$$L_w = \frac{\sum_{i=1}^{n} w_{y_i}\,\big(-\log p_{i, y_i}\big)}{\sum_{i=1}^{n} w_{y_i}},$$

where $p_{i, y_i}$ is the softmax probability of example $i$'s true class. Dividing by the sum of the weights, not by $n$, keeps the result on the scale of an average; frameworks such as PyTorch do the same for `CrossEntropyLoss(weight=...)` with its default mean reduction. A common choice is the **balanced** weight $w_c = n / (K\, n_c)$ for a class with $n_c$ of the $n$ rows and $K$ classes. With this normalisation only the *ratio* of the weights matters.

**Shapes.** `logits` is a `Tensor` of shape `(batch, K)`, `targets` an integer array `(batch,)`, `class_weights` a float array `(K,)`. The result is a scalar `Tensor`, so that `.backward()` can run through it. Weighting adds no new information about the rare class: it changes how much each example pulls on the weights.

In [ ]:
counts = np.bincount(y_train, minlength=2)
balanced = len(y_train) / (2 * counts)
print("class counts:", counts)
print("balanced weights:", balanced.round(3), "  ratio positive / negative:", round(balanced[1] / balanced[0], 2))

### ✍️ Your turn: `weighted_cross_entropy`

Write `weighted_cross_entropy(logits, targets, class_weights)` with engine operations, so that the gradient flows back to the logits:

- **inputs:** `logits`, a `Tensor` `(batch, K)`; `targets`, integers `(batch,)`; `class_weights`, a NumPy array `(K,)`;
- **output:** the scalar `Tensor` $L_w$ from the formula above.

Hints: `logits.log_softmax(axis=-1)` gives log-probabilities; indexing it with `[np.arange(len(targets)), targets]` picks each example's true-class log-probability and keeps it on the tape; `class_weights[targets]` is a plain NumPy array of per-example weights; `(tensor * array).sum()` stays a `Tensor`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def weighted_cross_entropy_reference(logits, targets, class_weights):
    log_probs = logits.log_softmax(axis=-1)
    picked = log_probs[np.arange(len(targets)), targets]
    weights = class_weights[targets]
    return -(picked * weights).sum() / weights.sum()

In [ ]:
def weighted_cross_entropy(logits, targets, class_weights):
    """Class-weighted mean cross-entropy as a scalar Tensor.
    logits: Tensor (batch, K); targets: ints (batch,); class_weights: array (K,)."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

The check below runs both versions on fresh tensors, calls `backward()`, and compares the loss **and** its gradient with respect to the logits. A version that leaves the tape (for example by using `.data`) gives the right number but no gradient, and the check will tell you.

In [ ]:
def loss_and_gradient(loss_function):
    """Wrap a loss so that practice() can compare its value and its gradient with respect to the logits."""
    def run(logits, targets, class_weights):
        z = Tensor(logits, requires_grad=True)
        loss = loss_function(z, targets, class_weights)
        loss.backward()
        return float(loss.data), z.grad
    return run


yours = loss_and_gradient(weighted_cross_entropy)
checked = practice("weighted_cross_entropy", yours, loss_and_gradient(weighted_cross_entropy_reference), cases=[
    (np.array([[2.0, -1.0], [0.5, 0.5], [-1.0, 3.0]]), np.array([0, 1, 1]), np.array([1.0, 4.0])),
    (np.random.default_rng(1).normal(size=(6, 3)), np.array([0, 1, 2, 2, 1, 0]), np.array([0.5, 1.0, 2.0])),
    (np.array([[1.0, 2.0], [3.0, -1.0]]), np.array([1, 0]), np.array([1.0, 1.0])),  # equal weights: plain cross-entropy
])
weighted_cross_entropy = weighted_cross_entropy if checked is yours else weighted_cross_entropy_reference

demo_logits = Tensor(np.array([[2.0, -1.0], [0.5, 0.5], [-1.0, 3.0]]))
demo_targets = np.array([0, 1, 1])
print(f"weights [1, 4]: {float(weighted_cross_entropy(demo_logits, demo_targets, np.array([1.0, 4.0])).data):.4f}   "
      f"plain mean: {float(cross_entropy(demo_logits, demo_targets).data):.4f}")

**Expected behaviour.** For the three examples above the per-example losses are $0.0486$, $0.6931$ and $0.0181$. Plain cross-entropy averages them to $0.2533$. With weights $[1, 4]$ the two positive examples count four times: $(0.0486 + 4 \cdot 0.6931 + 4 \cdot 0.0181) / 9 \approx 0.3215$. Multiplying both weights by the same constant leaves the loss unchanged, which is why only the ratio matters. (The engine's own `cross_entropy(logits, targets, mask=...)` accepts per-example weights in the same way, as long as they add up to at least 1.)

## 5 · Train a plain model and a weighted model

Two networks with the same architecture, the same initial weights, the same mini-batches and the same 400 updates. The only difference is the loss: plain cross-entropy, or your weighted cross-entropy with the balanced weights. `positive_probability` turns logits into the probability of the positive class.

In [ ]:
def positive_probability(model, X):
    """P(positive) for every row of X: a NumPy array of shape (n,)."""
    model.eval()
    return model(Tensor(X)).softmax(axis=-1).data[:, 1]


def rates_at(scores, y, threshold):
    """Recall and precision of the rule 'positive when score >= threshold'."""
    predicted = scores >= threshold
    tp = int(np.sum(predicted & (y == 1)))
    fp = int(np.sum(predicted & (y == 0)))
    return tp / max(int(y.sum()), 1), tp / max(tp + fp, 1)


w_balanced = len(y_train) / (2 * np.bincount(y_train))
plain = fit(MLP([6, 32, 2], np.random.default_rng(0)), X_train, y_train, steps=400, seed=1)
weighted = fit(MLP([6, 32, 2], np.random.default_rng(0)), X_train, y_train, steps=400, seed=1,
               loss_fn=lambda logits, targets: weighted_cross_entropy(logits, targets, w_balanced))

p_plain_val = positive_probability(plain, X_val)
p_weighted_val = positive_probability(weighted, X_val)
for name, p in [("plain", p_plain_val), ("weighted", p_weighted_val)]:
    recall, precision = rates_at(p, y_val, 0.5)
    print(f"{name:>8}: at threshold 0.5 recall {recall:.2f}, precision {precision:.2f}; "
          f"mean predicted probability {p.mean():.3f} (validation prevalence {y_val.mean():.3f})")

**What to notice.** At the default threshold of 0.5 the weighted model finds many more positives (higher recall) and raises more false alarms (lower precision). Its average predicted probability is also well above the true prevalence: weighting has made its probabilities over-confident about the positive class. Is the weighted model *better*, or does it just say "positive" more readily? A threshold-free view answers that.

## 6 · A precision–recall curve from scratch

For a threshold $t$, the rule "positive when score $\ge t$" produces true positives (TP), false positives (FP) and false negatives (FN):

$$\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN} = \frac{TP}{P},$$

where $P$ is the number of positive examples. Lowering $t$ can only add predicted positives, so recall never decreases while precision can move either way. Only the distinct score values matter as thresholds: every threshold between two neighbouring scores makes the same predictions.

**Shapes.** `scores` is a float array `(n,)` and `y` a 0/1 array `(n,)`. The result is three arrays of the same length $m$, the number of distinct scores: `thresholds` in **decreasing** order, and `precision[i]` and `recall[i]` for the rule `scores >= thresholds[i]`.

The demo counts by hand on six examples, including a tie at 0.8.

In [ ]:
scores_demo = np.array([0.9, 0.8, 0.8, 0.6, 0.3, 0.1])
y_demo = np.array([1, 1, 0, 0, 1, 0])
for t in [0.9, 0.8, 0.6, 0.3, 0.1]:
    predicted = scores_demo >= t
    tp, fp = np.sum(predicted & (y_demo == 1)), np.sum(predicted & (y_demo == 0))
    print(f"threshold {t}: TP={tp} FP={fp}  precision={tp / (tp + fp):.3f}  recall={tp / y_demo.sum():.3f}")

### ✍️ Your turn: `precision_recall_curve`

Write `precision_recall_curve(scores, y)`:

- **inputs:** `scores` `(n,)` and labels `y` `(n,)` of 0 and 1;
- **output:** a tuple `(thresholds, precision, recall)` of NumPy arrays, one entry per distinct score, thresholds in decreasing order. Compute recall as `TP / max(P, 1)` so that a set without positives gives 0 rather than a division by zero.

Hints: `np.unique(scores)[::-1]` gives the distinct scores from largest to smallest. A loop over them with the counting from the demo is perfectly fine for a few hundred rows. (A faster version sorts the scores once and uses `np.cumsum`; try it in the exercises if you like.)

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def precision_recall_curve_reference(scores, y):
    order = np.argsort(-scores, kind="stable")
    sorted_scores, sorted_y = scores[order], y[order]
    true_pos = np.cumsum(sorted_y)
    false_pos = np.cumsum(1 - sorted_y)
    last = np.r_[np.flatnonzero(np.diff(sorted_scores)), len(scores) - 1]  # last row of each run of equal scores
    tp, fp = true_pos[last], false_pos[last]
    return sorted_scores[last], tp / (tp + fp), tp / max(int(y.sum()), 1)

In [ ]:
def precision_recall_curve(scores, y):
    """Precision and recall for 'positive when score >= t', for every distinct score t (largest first).
    scores: (n,), y: (n,) of 0/1. Returns (thresholds, precision, recall), three arrays of equal length."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
tie_scores = np.round(np.random.default_rng(2).random(40), 1)  # many ties
tie_labels = (np.random.default_rng(3).random(40) < tie_scores).astype(int)
precision_recall_curve = practice("precision_recall_curve", precision_recall_curve, precision_recall_curve_reference, cases=[
    (scores_demo, y_demo),
    (tie_scores, tie_labels),
    (np.array([0.7, 0.2, 0.4]), np.array([0, 0, 0])),  # edge case: no positives at all
])

**Expected behaviour.** For the demo you should get thresholds `[0.9, 0.8, 0.6, 0.3, 0.1]`, precision `[1, 0.667, 0.5, 0.6, 0.5]` and recall `[0.333, 0.667, 0.667, 1, 1]`: the two tied scores at 0.8 enter together. Recall never decreases along the arrays, while precision goes down and up again.

Average precision summarises the whole curve in one number: the precision at each threshold, weighted by how much recall it adds, $\text{AP} = \sum_k (R_k - R_{k-1})\, P_k$. A classifier that ranks at random scores about the prevalence; a perfect ranking scores 1. The helper is given.

In [ ]:
def average_precision(precision, recall):
    """Area under the step-shaped precision-recall curve: sum of precision times the recall it adds."""
    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))


curves_val = {name: precision_recall_curve(p, y_val) for name, p in [("plain", p_plain_val), ("weighted", p_weighted_val)]}
plt.figure(figsize=(5.5, 4))
for name, (_, precision, recall) in curves_val.items():
    print(f"{name:>8}: average precision on validation {average_precision(precision, recall):.3f}")
    plt.step(recall, precision, where="post", label=name)
plt.axhline(y_val.mean(), color="grey", linestyle="--", label="random ranking")
plt.xlabel("recall"); plt.ylabel("precision"); plt.ylim(0, 1.02); plt.legend(); plt.title("validation precision-recall curves")
plt.show()

**What to notice.** The two curves are close and their average precisions differ by a few hundredths at most. Weighting barely changed how well the model **ranks** examples; it mostly changed which point of the curve the threshold 0.5 lands on. Moving along the curve is the job of the threshold.

## 7 · Choose the threshold on validation data, then test once

A decision rule needs a threshold. Choose it on **validation** data using an objective declared in advance, freeze it, and only then look at the test set, once. Choosing the threshold with test labels is model selection on the test set, even though no weight changes: the test score would then be optimistic.

Our objective is the **F1 score**, the harmonic mean of precision and recall:

$$F_1 = \frac{2PR}{P + R}.$$

It is high only when both are reasonably high. (Section 9 and the exercises show other objectives, such as a minimum recall or an explicit cost per error.)

### ✍️ Your turn: `choose_threshold`

Write `choose_threshold(scores, y)`:

- **inputs:** validation `scores` `(n,)` and labels `y` `(n,)`;
- **output:** `(threshold, f1)` as two Python floats: the threshold from `precision_recall_curve(scores, y)` with the highest F1. If several thresholds tie, return the **largest** one, which raises the fewest alarms.

Hints: compute F1 for all thresholds at once with `2 * p * r / np.maximum(p + r, 1e-12)` (the `np.maximum` avoids dividing by zero); `np.argmax` returns the first maximum, which is the largest threshold because the thresholds are in decreasing order.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def choose_threshold_reference(scores, y):
    thresholds, precision, recall = precision_recall_curve(scores, y)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    best = int(np.argmax(f1))
    return float(thresholds[best]), float(f1[best])

In [ ]:
def choose_threshold(scores, y):
    """The threshold with the best F1 (the largest one if several tie). Returns (threshold, f1) as floats."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
choose_threshold = practice("choose_threshold", choose_threshold, choose_threshold_reference, cases=[
    (scores_demo, y_demo),
    (tie_scores, tie_labels),
    (np.array([0.9, 0.7, 0.5, 0.3]), np.array([1, 0, 0, 1])),  # edge case: F1 ties at 0.9 and 0.3
])

**Expected behaviour.** On the demo the F1 scores are $0.5, 0.667, 0.571, 0.75, 0.667$, so you should get `(0.3, 0.75)`. In the third case thresholds 0.9 and 0.3 both give $F_1 = 2/3$, and the rule returns 0.9.

Now the protocol. Each model gets its own threshold from validation data; the thresholds are frozen; only then are the test probabilities computed and compared with the test labels, once.

In [ ]:
def confusion_counts(scores, y, threshold):
    """TP, FP, FN and TN of the rule 'positive when score >= threshold'."""
    predicted = scores >= threshold
    return {"tp": int(np.sum(predicted & (y == 1))), "fp": int(np.sum(predicted & (y == 0))),
            "fn": int(np.sum(~predicted & (y == 1))), "tn": int(np.sum(~predicted & (y == 0)))}


t_plain, f1_plain_val = choose_threshold(p_plain_val, y_val)
t_weighted, f1_weighted_val = choose_threshold(p_weighted_val, y_val)
print(f"validation-chosen thresholds: plain {t_plain:.3f} (F1 {f1_plain_val:.3f}), weighted {t_weighted:.3f} (F1 {f1_weighted_val:.3f})")

# The thresholds are frozen. Only now do we touch the test set, once.
p_plain_test = positive_probability(plain, X_test)
p_weighted_test = positive_probability(weighted, X_test)
test_results = {}
print(f"\n{'model':>8} {'rule':>16} {'TP':>4} {'FP':>4} {'FN':>4} {'precision':>9} {'recall':>6} {'F1':>6}")
for model_name, p_test, chosen in [("plain", p_plain_test, t_plain), ("weighted", p_weighted_test, t_weighted)]:
    for rule, t in [("threshold 0.5", 0.5), ("validation F1", chosen)]:
        c = confusion_counts(p_test, y_test, t)
        precision, recall = c["tp"] / max(c["tp"] + c["fp"], 1), c["tp"] / max(c["tp"] + c["fn"], 1)
        f1 = 2 * c["tp"] / max(2 * c["tp"] + c["fp"] + c["fn"], 1)
        test_results[(model_name, rule)] = f1
        print(f"{model_name:>8} {rule:>16} {c['tp']:4d} {c['fp']:4d} {c['fn']:4d} {precision:9.2f} {recall:6.2f} {f1:6.3f}")
print(f"\nThe test set has {y_test.sum()} positives: one more or one fewer true positive moves recall by {1 / y_test.sum():.3f}.")

**What to notice.** The weighted model's chosen threshold, about 0.92, sits far above the plain model's, about 0.65: weighting moved the threshold that works best. At the fixed threshold of 0.5 the two models make very different decisions, but once each has its own validated threshold their test F1 scores are almost the same. With fewer than 60 test positives, differences of a few hundredths are within the noise of a single test set, so report the counts rather than declaring a winner.

## 8 · Ranking, probabilities and decisions

The weighted model's probabilities are inflated in a predictable way. With weights $w_0$ and $w_1$, minimising the weighted loss pushes the model's odds of *positive* up by the ratio $w = w_1 / w_0$. Two consequences follow, and the cell checks both:

- a threshold $t$ on the plain model corresponds to $\dfrac{w t}{w t + 1 - t}$ on the weighted model, so 0.5 becomes $w / (1 + w)$;
- dividing the weighted model's odds by $w$, that is $\hat p = \dfrac{p}{p + w(1 - p)}$, gives probabilities back on the scale of the real prevalence.

The correction is monotone, so it changes the probabilities without changing the ranking or the precision–recall curve. If probabilities themselves drive decisions, check their calibration on held-out data.

In [ ]:
ratio = w_balanced[1] / w_balanced[0]
corrected = p_weighted_val / (p_weighted_val + ratio * (1 - p_weighted_val))
print(f"weight ratio w = {ratio:.2f}; threshold 0.5 on the plain model corresponds to {ratio / (1 + ratio):.3f} on the weighted one")
print(f"the weighted model's validation-chosen threshold was {t_weighted:.3f}")
print(f"mean probability on validation: plain {p_plain_val.mean():.3f}, weighted {p_weighted_val.mean():.3f}, "
      f"weighted after correction {corrected.mean():.3f}")
print(f"prevalence: training {y_train.mean():.3f}, validation {y_val.mean():.3f}")
_, precision_c, recall_c = precision_recall_curve(corrected, y_val)
print(f"average precision before and after the correction: {average_precision(*curves_val['weighted'][1:]):.3f} and "
      f"{average_precision(precision_c, recall_c):.3f}")

**What to notice.** The weighted model's validation-chosen threshold lands close to the predicted $w/(1+w)$. The correction brings its mean probability back to the plain model's level, near the training prevalence that both models learned from, and leaves the average precision unchanged. Weighting did not teach the model anything new about the rare class; it shifted the model's probabilities, and with them the threshold that suits them.

## 9 · 🧪 Experiments

### 🧪 How strong should the weight be?

Change `positive_weight` and run the cell again. Watch four numbers: recall at 0.5, the validation-chosen threshold, the best validation F1 and the average precision.

In [ ]:
positive_weight = 8.0  # 🧪 try 1.0, 3.0 and 30.0, then run this cell again
model = fit(MLP([6, 32, 2], np.random.default_rng(0)), X_train, y_train, steps=400, seed=1,
            loss_fn=lambda logits, targets: weighted_cross_entropy(logits, targets, np.array([1.0, positive_weight])))
p = positive_probability(model, X_val)
recall, precision = rates_at(p, y_val, 0.5)
t_best, f1_best = choose_threshold(p, y_val)
_, prec, rec = precision_recall_curve(p, y_val)
print(f"positive weight {positive_weight:g}: recall at 0.5 {recall:.2f}, precision at 0.5 {precision:.2f}, "
      f"chosen threshold {t_best:.3f}, best F1 {f1_best:.3f}, average precision {average_precision(prec, rec):.3f}")

As the weight grows from 1 to 30, recall at 0.5 rises from about 0.83 to 0.96 and the chosen threshold climbs from about 0.65 towards 1, while the best F1 and the average precision change much less. Very large weights can make training noisier, because a handful of positive examples then dominate each update.

### 🧪 Capacity or data?

Change the hidden width and the number of training rows, then read the diagnosis from the two losses and the baseline.

In [ ]:
hidden_width = 32   # 🧪 try 2 and 128
training_rows = 1000  # 🧪 try 100 and 300
model = fit(MLP([6, hidden_width, 2], np.random.default_rng(0)), X_train[:training_rows], y_train[:training_rows], steps=300)
train_loss, val_loss = mean_loss(model, X_train[:training_rows], y_train[:training_rows]), mean_loss(model, X_val, y_val)
print(f"width {hidden_width}, {training_rows} rows: training loss {train_loss:.3f}, validation loss {val_loss:.3f}, baseline {baseline:.3f}")
linear_reference = curves["linear"][-1, 2]  # the linear model with 1,000 rows, from Section 3
if val_loss - train_loss > 0.1:
    print("a large gap: the model memorises these rows (variance); add rows, regularise or stop earlier")
elif val_loss > linear_reference - 0.02:
    print(f"a small gap, but no better than the linear model ({linear_reference:.3f}): too little capacity (bias)")
else:
    print("a small gap and well below the linear model: this model is not the bottleneck")

## 10 · Put it together

The cell below re-checks the whole protocol on behaviour, not on exact numbers: the MLP beats both the no-skill baseline and the linear model; weighting raises recall at 0.5 and inflates probabilities without changing the ranking much; and both decision rules, chosen on validation and frozen, reach a useful F1 on the untouched test set.

In [ ]:
mlp_val_loss = mean_loss(plain, X_val, y_val)
linear_val_loss = curves["linear"][-1, 2]
ap_plain = average_precision(*curves_val["plain"][1:])
ap_weighted = average_precision(*curves_val["weighted"][1:])

assert mlp_val_loss < 0.6 * baseline, f"the MLP should beat the no-skill baseline clearly ({mlp_val_loss:.3f} vs {baseline:.3f})"
assert mlp_val_loss < 0.9 * linear_val_loss, "the hidden layer should beat the linear model: the interaction needs it"
assert history[-1, 2] > best_row[2] + 0.1, "the wide model on 150 rows should overfit after its early-stopping point"
assert min(ap_plain, ap_weighted) > 3 * y_val.mean(), "both models should rank positives far better than chance"
assert abs(ap_plain - ap_weighted) < 0.1, "weighting should barely change the ranking"
assert rates_at(p_weighted_val, y_val, 0.5)[0] > rates_at(p_plain_val, y_val, 0.5)[0] + 0.02, "weighting should raise recall at 0.5"
assert p_weighted_val.mean() > p_plain_val.mean() + 0.02, "weighting should inflate the predicted probabilities"
assert t_weighted > t_plain, "weighting should move the best threshold up"
assert min(test_results[("plain", "validation F1")], test_results[("weighted", "validation F1")]) > 0.55, "test F1 is too low"
print("✅ Diagnosis, weighting and threshold selection all behave as expected.")
print(f"   validation loss: no-skill {baseline:.3f}, linear {linear_val_loss:.3f}, MLP {mlp_val_loss:.3f}")
print(f"   average precision: plain {ap_plain:.3f}, weighted {ap_weighted:.3f}")
print(f"   test F1 at the frozen thresholds: plain {test_results[('plain', 'validation F1')]:.3f}, "
      f"weighted {test_results[('weighted', 'validation F1')]:.3f}")

## Exercises

These go beyond the type-alongs. The first two have checks; the last two are questions to answer in a markdown cell of your own.

### Exercise 1: a minimum-recall rule

Suppose missing a positive is expensive, and the requirement is "find at least 80% of the positives". Write `threshold_for_recall(scores, y, min_recall)` that returns `(threshold, precision, recall)` for the **largest** threshold whose recall is at least `min_recall`. The largest such threshold raises the fewest alarms. Notebook 4's capstone uses this rule.

In [ ]:
def threshold_for_recall_reference(scores, y, min_recall):
    thresholds, precision, recall = precision_recall_curve(scores, y)
    first = int(np.flatnonzero(recall >= min_recall)[0])  # recall never decreases along the arrays
    return float(thresholds[first]), float(precision[first]), float(recall[first])

In [ ]:
def threshold_for_recall(scores, y, min_recall):
    """The largest threshold with recall >= min_recall. Returns (threshold, precision, recall) as floats."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
threshold_for_recall = practice("threshold_for_recall", threshold_for_recall, threshold_for_recall_reference, cases=[
    (scores_demo, y_demo, 0.6),
    (p_plain_val, y_val, 0.8),
    (scores_demo, y_demo, 1.0),  # edge case: every positive must be found
])

### Exercise 2: a threshold from costs

A false negative costs `cost_fn` and a false positive costs `cost_fp`. Write `cost_threshold(scores, y, cost_fn, cost_fp)` that returns `(threshold, total_cost)` for the threshold from `precision_recall_curve` with the lowest total cost `cost_fn * FN + cost_fp * FP` on the given data, the largest threshold if several tie. Hint: loop over the thresholds and count FN and FP as in the demo of Section 6; `np.argmin` returns the first minimum.

In [ ]:
def cost_threshold_reference(scores, y, cost_fn, cost_fp):
    thresholds, _, _ = precision_recall_curve(scores, y)
    costs = []
    for t in thresholds:
        predicted = scores >= t
        false_neg = np.sum(~predicted & (y == 1))
        false_pos = np.sum(predicted & (y == 0))
        costs.append(cost_fn * false_neg + cost_fp * false_pos)
    best = int(np.argmin(costs))
    return float(thresholds[best]), float(costs[best])

In [ ]:
def cost_threshold(scores, y, cost_fn, cost_fp):
    """The threshold with the lowest total cost (the largest one if several tie). Returns (threshold, cost)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
cost_threshold = practice("cost_threshold", cost_threshold, cost_threshold_reference, cases=[
    (scores_demo, y_demo, 5.0, 1.0),
    (p_plain_val, y_val, 10.0, 1.0),
    (scores_demo, y_demo, 1.0, 1.0),
])

### Exercise 3 (question): where should the weighted threshold be?

Section 8 claimed that the weighted model's odds are the plain model's odds multiplied by $w$. Use that claim to predict the weighted model's best threshold from the plain model's `t_plain`, and compare your prediction with `t_weighted`. Why do they not agree exactly? (Think about two separately trained networks and about the 50 or so validation positives.)

### Exercise 4 (question): how stable is the threshold?

Generate a fresh validation set with another seed, say `make_rare_class_data(500, seed=44)`, standardise it with the **training** `mean` and `scale`, and choose the threshold again. How far does it move, and how much does the validation F1 change? What does this tell you about reporting a threshold to three decimal places?

## Common errors and debugging

- **Fitting the standardiser on every row.** Validation and test statistics then leak into training. Fit on the training rows and apply the same `mean` and `scale` everywhere.
- **Reporting accuracy on a rare class.** A model that never predicts *positive* scores about 89% here. Report precision, recall, the confusion counts and the number of positives.
- **A softmax before the loss.** `cross_entropy` and your weighted loss expect raw logits; applying softmax first squashes the gradients.
- **Dividing a weighted loss by $n$ instead of by the sum of the weights.** Both conventions exist (PyTorch's `BCEWithLogitsLoss(pos_weight=...)` divides by $n$), but they differ by a factor that acts like a different learning rate. Know which one you use.
- **A precision–recall curve that starts at recall 1.** The thresholds are in increasing order. Sort them from largest to smallest.
- **`RuntimeWarning: invalid value encountered in divide`.** Precision plus recall is zero at some threshold; guard the division with `np.maximum(..., 1e-12)`.
- **Choosing the threshold, the checkpoint or the weighting on the test set.** Every such choice belongs to validation data; the test set is opened once, at the end.

## Summary

A baseline and two learning curves told a capacity problem (the linear model: both losses high and flat) from a data problem (the small-data MLP: a large gap that closes as rows are added). On a rare class, you wrote class-weighted cross-entropy and saw that it raises recall at a fixed threshold, inflates the predicted probabilities and leaves the ranking almost unchanged. The precision–recall curve and its average precision measure the ranking; the threshold, chosen on validation data and frozen, turns the ranking into decisions; the test set, opened once, measures those decisions. Remember that these are small teaching models on generated data: real systems add messier labels, shifting prevalence and far more rows, and a threshold chosen here says nothing about them.

## Next

**02 · Optimisation and transfer**: control how gradients become updates, with a warm-up and cosine learning-rate schedule, gradient clipping and gradient accumulation, then reuse a pretrained network for a new task with very few labels.